In [3]:
import numpy as np
import random
import os
import tensorflow as tf
# On importe explicitement Input ici pour éviter le NameError
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input 
from tensorflow.keras.optimizers import Adam
from collections import deque
import matplotlib.pyplot as plt
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

In [9]:
# Paramètres du jeu 
GRID_SIZE = 4 
STATE_SIZE = GRID_SIZE * GRID_SIZE 
ACTION_SIZE = 4  # Haut, Bas, Gauche, Droite 
GAMMA = 0.9  # facteur de reduction  
LEARNING_RATE = 0.01 
EPSILON = 1.0  # Exploration initiale 
EPSILON_MIN = 0.01 
EPSILON_DECAY = 0.995 
BATCH_SIZE = 32 
MEMORY_SIZE = 2000 
EPISODES = 1000 

# Déplacements possibles (Haut, Bas, Gauche, Droite) 
MOVES = { 
0: (-1, 0),  # Haut 
1: (1, 0),   # Bas 
2: (0, -1),  # Gauche 
3: (0, 1)    
# Droite 
}

In [10]:
class GridWorld: 
    """Environnement GridWorld 4x4""" 
    def __init__(self): 
        self.grid_size = GRID_SIZE 
        self.reset() 
 
    def reset(self): 
        """Réinitialise l'agent à la position de départ.""" 
        self.agent_pos = (0, 0) 
        self.goal_pos = (3, 3) 
        self.obstacle_pos = (1, 1)  # Une case d'obstacle 
        return self.get_state() 
 
    def get_state(self): 
        """Retourne l'état sous forme d'un vecteur binaire.""" 
        state = np.zeros((GRID_SIZE, GRID_SIZE)) 
        state[self.agent_pos] = 1 
        return state.flatten() 
 
    def step(self, action): 
        """Fait avancer l'agent et renvoie (nouvel état, récompense, 
terminé).""" 
        x, y = self.agent_pos 
        dx, dy = MOVES[action] 
        new_x, new_y = x + dx, y + dy 
 
        # Vérifier les limites 
        if 0 <= new_x < GRID_SIZE and 0 <= new_y < GRID_SIZE: 
            self.agent_pos = (new_x, new_y) 
 
        # Vérifier la récompense 
        if self.agent_pos == self.goal_pos: 
            return self.get_state(), 10, True  # Objectif atteint  
        elif self.agent_pos == self.obstacle_pos: 
            return self.get_state(), -5, False  # Obstacle  
        else: 
            return self.get_state(), -1, False  # Déplacement normal

In [13]:
class DoubleDQNAgent:
    def __init__(self):
        self.state_size = STATE_SIZE
        self.action_size = ACTION_SIZE
        self.memory = deque(maxlen=MEMORY_SIZE) # [cite: 92]
        self.epsilon = EPSILON
        
        # Double Réseau : Online et Target [cite: 169]
        self.model = self._build_model()
        self.target_model = self._build_model()
        self.update_target_network()

    def _build_model(self):
        # Utilisation de Input() pour éviter les avertissements 
        model = Sequential([
            Input(shape=(self.state_size,)),
            Dense(24, activation='relu'),
            Dense(24, activation='relu'),
            Dense(self.action_size, activation='linear')
        ])
        model.compile(loss="mse", optimizer=Adam(learning_rate=LEARNING_RATE))
        return model

    def update_target_network(self):
        """Synchronisation des poids [cite: 167]"""
        self.target_model.set_weights(self.model.get_weights())

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state):
        if np.random.rand() <= self.epsilon: # [cite: 106]
            return random.randrange(self.action_size)
        q_values = self.model.predict(np.array([state]), verbose=0)
        return np.argmax(q_values[0])

    def replay(self):
        if len(self.memory) < BATCH_SIZE:
            return
        
        batch = random.sample(self.memory, BATCH_SIZE)
        
        for state, action, reward, next_state, done in batch:
            target = self.model.predict(np.array([state]), verbose=0)[0]
            if done:
                target[action] = reward
            else:
                # Logique Double DQN [cite: 175, 176]
                next_q_online = self.model.predict(np.array([next_state]), verbose=0)[0]
                best_action = np.argmax(next_q_online)
                
                next_q_target = self.target_model.predict(np.array([next_state]), verbose=0)[0]
                target[action] = reward + GAMMA * next_q_target[best_action]

            self.model.fit(np.array([state]), np.array([target]), epochs=1, verbose=0)

        if self.epsilon > EPSILON_MIN:
            self.epsilon *= EPSILON_DECAY

In [2]:
env = GridWorld()
agent = DoubleDQNAgent()
scores = []

for episode in range(EPISODES):
    state = env.reset()
    total_reward = 0
    
    for step in range(50): 
        action = agent.act(state)
        next_state, reward, done = env.step(action)
        agent.remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
        if done:
            break
            
    agent.replay()
    
    # Mise à jour du réseau cible tous les TARGET_UPDATE_FREQ épisodes [cite: 167]
    if (episode + 1) % TARGET_UPDATE_FREQ == 0:
        agent.update_target_network()
        
    scores.append(total_reward)
    if (episode + 1) % 100 == 0:
        print(f"Épisode {episode+1}/{EPISODES}, Score Moyen: {np.mean(scores[-100:]):.2f}, Epsilon: {agent.epsilon:.4f}")

# Sauvegarde finale [cite: 160]
agent.model.save("double_dqn_model.keras")

NameError: name 'GridWorld' is not defined

In [5]:
# Initialisation
env = GridWorld()
agent = DoubleDQNAgent()

# Pour stocker les métriques
scores = []
epsilon_history = []
target_updates = 0

print("\n" + "="*60)
print("DÉBUT DE L'ENTRAÎNEMENT - DOUBLE DQN")
print("="*60)

for episode in range(EPISODES):
    state = env.reset()
    total_reward = 0
    steps = 0
    
    # Limite de 50 déplacements par épisode
    for step in range(50):
        action = agent.act(state)
        next_state, reward, done = env.step(action)
        agent.remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
        steps += 1
        if done:
            break
    
    # Entraînement sur les expériences
    agent.replay()
    scores.append(total_reward)
    epsilon_history.append(agent.epsilon)
    
    # Mise à jour périodique du réseau cible
    if (episode + 1) % TARGET_UPDATE_FREQ == 0:
        agent.update_target()
        target_updates += 1
    
    # Affichage des progrès
    if (episode + 1) % 100 == 0 or episode == 0:
        avg_score = np.mean(scores[-100:]) if len(scores) >= 100 else np.mean(scores)
        print(f"Épisode {episode+1:4d}/{EPISODES} | "
              f"Score: {total_reward:6.2f} | "
              f"Moyenne(100): {avg_score:6.2f} | "
              f"Epsilon: {agent.epsilon:.4f} | "
              f"Pas: {steps:2d}")

print("\n" + "="*60)
print("ENTRAÎNEMENT TERMINÉ!")
print(f"  - Total épisodes: {EPISODES}")
print(f"  - Mises à jour du réseau cible: {target_updates}")
print(f"  - Epsilon final: {agent.epsilon:.4f}")
print("="*60)

# Sauvegarde du modèle
agent.online_network.save("double_dqn_model.keras")
print("Modèle sauvegardé sous 'double_dqn_model.keras'")

NameError: name 'cite' is not defined

In [ ]:
plt.plot(scores)
plt.title("Évolution des scores (Double DQN)")
plt.xlabel("Épisodes")
plt.ylabel("Récompense")
plt.show()